<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.4: A Single-Cycle RV32I Processor
**Prev: [The Decoder](5.3_decoder.ipynb)**<br>
**Next: [The Official RISC-V Tests](5.5_compliance.ipynb)**<br>

## Motivation
Everything is in place: the ALU and the register file ([5.2](5.2_alu_and_regfile.ipynb)), the
immediate generator, and the decoder ([5.3](5.3_decoder.ipynb)). This notebook connects them into
a processor that runs RV32I programs: a **single-cycle** one, which fetches, decodes, executes
and retires one instruction in each clock cycle. It is the simplest correct design, and the
reference for the faster ones that follow.

The processor uses your ALU and decode table: if you have not completed the exercises of 5.2 and
5.3 yet, do that first (the cells below stop with a `NotImplementedError` until then). This
notebook adds two more exercises: the branch comparator and the load extension.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# Memory

The core talks to memory through two ports, one for instructions and one for data, of the same
type, `riscv.MemPort`: a request (`Decoupled`: address, write data, byte strobes) and a response
(`Valid`: the word read, or an error for an address outside the memory). The RAM of this notebook
answers in the same cycle, because it is a `Mem` (combinational read, [Module 2.8](2.8_memories.ipynb)):
a single-cycle core must fetch and load within one cycle. This core relies on that: it retires an
instruction only in a cycle where both answers are there, and ignores `ready`. A memory that
answers later, such as a bus, needs a core that keeps track of its requests (Module 5.11).

Memory is byte-addressed, but the RAM stores words: a store of a byte or a half-word sends the
word with the bytes in place, and *strobes* that say which bytes to write. The RAM has one more
port, `load`, through which a testbench writes the program before the core starts.

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "Memory.scala"))

Note the comment about the write port: `firtool` drops the write masks of a memory that also has
a write port without a mask (see the end of the write-mask section of
[Module 2.8](2.8_memories.ipynb)), so the RAM has one write port, which the load port and the
stores share.

---
# The datapath

`riscv.SingleCycleCore` follows an instruction through the cycle:

1. **fetch** the word at `pc` from the instruction port;
2. **decode** it (`Decoder`), read `rs1` and `rs2` (`RegFile`), and build the immediate (`ImmGen`);
3. **execute**: the ALU computes the result, a memory address or a jump target, with the operands
   that `aSel` and `bSel` choose; the **branch unit** compares `rs1` and `rs2`;
4. **memory**: a load or a store goes to the data port; a load's byte or half-word is extracted
   and extended (`LoadExtend`);
5. **write back** the result (the ALU's, the loaded value, or pc + 4 for `jal`/`jalr`) to `rd`, and
   move to the next pc (pc + 4, or the target of a jump or a taken branch).

The core also checks what halts it (an illegal instruction, `ecall`, `ebreak`, a misaligned
access or target, an address outside the RAM) and reports every retired instruction on its
**retire port**, with the same fields as the golden model's `Retire` records: tests compare the
two streams instruction by instruction.

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "SingleCycleCore.scala"))

---
# Exercise 1: the branch comparator

A branch is taken if its two registers compare as its `funct3` says: `beq` (0), `bne` (1), `blt`
(4), `bge` (5), `bltu` (6), `bgeu` (7). Complete `riscv/src/main/scala/riscv/BranchUnit.scala`.
A hint: the low bit of `funct3` negates the condition (`bne` is not `beq`, `bge` is not `blt`).

In [ ]:
simulate(new BranchUnit) { c =>
  val rnd              = new scala.util.Random(1)
  def unsigned(x: Int) = BigInt(Integer.toUnsignedLong(x))
  val corners          = Seq(0, 1, -1, Int.MaxValue, Int.MinValue)
  def operand          = if (rnd.nextInt(3) == 0) corners(rnd.nextInt(corners.size)) else rnd.nextInt()
  for ((f3, name) <- Seq(0 -> "beq", 1 -> "bne", 4 -> "blt", 5 -> "bge", 6 -> "bltu", 7 -> "bgeu"); _ <- 0 until 300) {
    val (a, b) = if (rnd.nextInt(5) == 0) { val x = operand; (x, x) }
    else (operand, operand)
    c.io.funct3.poke(f3.U)
    c.io.a.poke(unsigned(a).U)
    c.io.b.poke(unsigned(b).U)
    c.io.taken.expect(IsaSim.taken(name, a, b).B, s"$name $a, $b") // the golden model's decision
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  // funct3 bit 0 negates the condition: bne, bge and bgeu are the opposites of beq, blt and bltu
  val condition = MuxLookup(io.funct3(2, 1), false.B)(
    Seq(
      0.U -&gt; (io.a === io.b),
      2.U -&gt; (io.a.asSInt &lt; io.b.asSInt),
      3.U -&gt; (io.a &lt; io.b)
    )
  )
  io.taken := condition =/= io.funct3(0)
</pre></article></div></section></div>

---
# Exercise 2: the load extension

The data port returns the whole word at the (word-aligned) address. For `lb`, `lbu`, `lh` and
`lhu`, the byte or half-word starts at byte `offset` of the word (the address's low two bits;
RISC-V is little-endian, so byte 0 is bits 7 to 0), and must be sign-extended (`lb`, `lh`) or
zero-extended (`lbu`, `lhu`) to 32 bits. Complete `riscv/src/main/scala/riscv/LoadExtend.scala`.

In [ ]:
simulate(new LoadExtend) { c =>
  val rnd              = new scala.util.Random(2)
  def unsigned(x: Int) = BigInt(Integer.toUnsignedLong(x))
  for (_ <- 0 until 300; (f3, name) <- Seq(0 -> "lb", 1 -> "lh", 2 -> "lw", 4 -> "lbu", 5 -> "lhu")) {
    val bytes  = IsaSim.width(name)
    val word   = rnd.nextInt()
    val offset = rnd.nextInt(4 / bytes) * bytes
    // the bytes the load reads, and what the golden model makes of them
    val raw = (Integer.toUnsignedLong(word) >>> (8 * offset)) & ((1L << (8 * bytes)) - 1)
    c.io.funct3.poke(f3.U)
    c.io.offset.poke(offset.U)
    c.io.rdata.poke(unsigned(word).U)
    c.io.value.expect(unsigned(IsaSim.loadValue(name, raw)).U, f"$name offset $offset word 0x$word%08x")
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val shifted  = io.rdata &gt;&gt; (io.offset ## 0.U(3.W)) // offset * 8
  val unsigned = io.funct3(2)
  val byte     = shifted(7, 0)
  val half     = shifted(15, 0)
  io.value := MuxLookup(io.funct3(1, 0), shifted)(
    Seq(
      0.U -&gt; Cat(Fill(24, byte(7) &amp;&amp; !unsigned), byte),
      1.U -&gt; Cat(Fill(16, half(15) &amp;&amp; !unsigned), half)
    )
  )
</pre></article></div></section></div>

---
# Running a program

`riscv.SingleCycleSystem` is the core with its RAM. `CoSim.load` writes a program (assembled by
`riscv.Asm`, [Module 5.1](5.1_riscv_isa.ipynb)) through the load port, `CoSim.run` clocks the
core until it halts and collects what it retires, and `CoSim.reg` reads a register through the
debug port. Here is the Fibonacci program, instruction by instruction:

In [ ]:
val fib = Asm(os.read(os.pwd / "riscv" / "programs" / "src" / "fib.S"))
simulate(new SingleCycleSystem) { c =>
  CoSim.load(c, fib)
  val trace = CoSim.run(c)
  for (r <- trace.take(12)) {
    val write = if (r.rd != 0) f"x${r.rd}%-2d = ${r.rdValue}" else ""
    println(f"${r.pc}%08x  ${Rv32i.decode(r.inst).map(_.toString).getOrElse("?")}%-28s $write")
  }
  println(s"... ${trace.size} instructions in ${trace.size} cycles, halted by ${trace.last.halt.get}")
  assert(CoSim.reg(c, riscv.Reg("a0")) == 55)
}
println("SUCCESS!!")

---
# Comparing with the golden model

A program that ends with the right result can still have taken a wrong turn on the way. The
strong test compares every retired instruction with the golden model, `IsaSim`:
`CoSim.compare` runs a program on both and returns the first difference. Here: the factorial
program (calls, the stack, and a multiplication by shifts and adds) and random programs
(`RandomProgram`, which are written so that they always end with `ecall`).

In [ ]:
simulate(new SingleCycleSystem) { c =>
  val fact = Asm(os.read(os.pwd / "riscv" / "programs" / "src" / "fact.S"))
  CoSim.compare(c, fact).foreach(d => sys.error(s"factorial: $d"))
  for (seed <- 1L to 5L)
    CoSim.compare(c, Asm(RandomProgram(seed))).foreach(d => sys.error(s"RandomProgram($seed): $d"))
}
println("SUCCESS!!")

---
# Halting

Without CSRs there is no trap handler: the instructions that would trap halt the core instead,
which reports why. An instruction that halts the core changes nothing (here, `ra` is not written
by the jump to a misaligned address):

In [ ]:
val programs = Seq(
  "ecall"                            -> Halt.Ecall,
  "ebreak"                           -> Halt.Ebreak,
  ".word 0x30200073  # mret"         -> Halt.Illegal,
  "la ra, x\njalr ra, 2(ra)\nx: nop" -> Halt.Misaligned,
  "li t0, 4\nlw a0, 0(t0)"           -> Halt.BadAddress
)
simulate(new SingleCycleSystem) { c =>
  for ((source, cause) <- programs) {
    CoSim.load(c, Asm(source))
    val last = CoSim.run(c).last
    println(f"${source.replace("\n", "; ")}%-40s halts: ${last.halt.get}")
    assert(last.halt.contains(cause) && last.rd == 0)
    if (cause == Halt.Misaligned) assert(CoSim.reg(c, 1) == 0x8000000cL, "ra still points to x") // not written
  }
}
println("SUCCESS!!")

---
# You're done!

The core runs one instruction per cycle, so its clock can only be as fast as the slowest
instruction's path (a load: fetch, decode, register read, ALU, memory, extension, write back).
The [next notebook](5.5_compliance.ipynb) runs the official RISC-V tests on it; later notebooks split the path into a
pipeline.

[Return to the top.](#top)